# Footage Engine — YouTube ingest

Clones the repo, reconstructs `.env` from Kaggle secrets, pulls YouTube cookies, and ingests one or more videos into scene chunks.

## Secrets to attach (Add-ons → Secrets)

Required:

| Secret | Purpose |
|---|---|
| `DATABASE_URL` | Shared job/media DB. **Must not be SQLite** — a Kaggle session's local file vanishes when it ends. |
| `VECTOR_STORE` | `zilliz`. A remote worker with `in_memory` finds nothing. |
| `ZILLIZ_URI`, `ZILLIZ_TOKEN`, `ZILLIZ_COLLECTION_NAME` | X-CLIP collection (512d). |
| `INGEST_URLS` | Comma-separated YouTube URLs. |
| `YT_COOKIES_URL` | HTTPS endpoint returning a Netscape `cookies.txt`. Never hardcode this — see the security note below. |

Required only when `EMBEDDING_BACKEND=qwen`:

| Secret | Purpose |
|---|---|
| `QWEN_ZILLIZ_URI`, `QWEN_ZILLIZ_TOKEN`, `QWEN_ZILLIZ_COLLECTION_NAME` | Qwen collection (2048d — cannot share the X-CLIP collection). |

Required only when `STORAGE_BACKEND=gdrive`:

| Secret | Purpose |
|---|---|
| `GDRIVE_OAUTH_CREDENTIALS_JSON` | Whole `authorized_user` token JSON, minified onto one line. |
| `GDRIVE_FOLDER_ID` | Destination folder. |
| `UPLOAD_RAW_TO_STORAGE`, `UPLOAD_CHUNKS_TO_STORAGE` | `true` to upload. |

## Security

This kernel is set `is_private: true`. It still clones a public GitHub repo and writes secrets into a working tree on Kaggle's filesystem, so:

- Never put a cookie URL, token, or key in a committed file. Everything comes from `kaggle_secrets`.
- `kernel-metadata.json` and this notebook are safe to commit; the values are not.
- Kaggle secrets are readable by **anyone who can open the kernel**. Keep it private and grant access only to collaborators.

In [ ]:
# !pip install -q --upgrade pip
# Kaggle ships torch + cv2 already; only add what is genuinely missing so
# the install does not pull a second multi-GB torch wheel over the pinned one.
import importlib, subprocess, sys

NEEDED = [
    ("transformers", "transformers>=4.46,<4.58"),
    ("sentence_transformers", "sentence-transformers"),
    ("scenedetect", "scenedetect[opencv]"),
    ("pymilvus", "pymilvus"),
    ("sqlalchemy", "sqlalchemy"),
    ("pydantic_settings", "pydantic-settings"),
    ("pydantic", "pydantic>=2.5"),
    ("psycopg2", "psycopg2-binary"),
    ("PIL", "pillow"),
    ("cv2", "opencv-python-headless"),
    ("imagekitio", "imagekitio"),
    ("yt_dlp", "yt-dlp"),
    ("dotenv", "python-dotenv"),
    ("requests", "requests"),
    ("googleapiclient", "google-api-python-client"),
    ("google.oauth2", "google-auth"),
]

missing = [spec for module, spec in NEEDED if importlib.util.find_spec(module) is None]
if missing:
    print(f"installing {len(missing)}: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies already present")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Rebuild .env from Kaggle secrets. Values never appear in the notebook or in git.
import os
from kaggle_secrets import UserSecretClient

secrets = UserSecretClient()

# name -> default written when the secret is absent (so an omitted optional
# secret degrades to a documented default instead of an empty value).
ENV_DEFAULTS = {
    "VECTOR_STORE": "zilliz",
    "EMBEDDING_BACKEND": "xclip",
    "EMBEDDING_DIMENSION": "512",
    "DEFAULT_EMBEDDING_MODEL": "microsoft/xclip-base-patch32",
    "EMBEDDING_DEVICE": "auto",
    "NUM_WORKERS": "2",
    "STORAGE_BACKEND": "local",
    "UPLOAD_RAW_TO_STORAGE": "false",
    "UPLOAD_CHUNKS_TO_STORAGE": "false",
    "GDRIVE_SCOPES": "https://www.googleapis.com/auth/drive",
    "GDRIVE_URL_TEMPLATE": "https://lh3.googleusercontent.com/d/{file_id}",
    "CHUNK_THRESHOLD_SEC": "45.0",
}

HARD_REQUIRED = [
    "DATABASE_URL",
    "VECTOR_STORE",
    "ZILLIZ_URI",
    "ZILLIZ_TOKEN",
    "ZILLIZ_COLLECTION_NAME",
    "INGEST_URLS",
    "YT_COOKIES_URL",
]

lines, missing = [], []
for name in HARD_REQUIRED:
    try:
        value = secrets.get_secret(name)
    except Exception:
        missing.append(name)
        continue
    lines.append(f"{name}={value}")
    os.environ[name] = value

if missing:
    raise SystemExit(
        "Missing required Kaggle secrets: " + ", ".join(missing)
        + "\nAttach them under Add-ons -> Secrets, then re-run this kernel."
    )

for name, default in ENV_DEFAULTS.items():
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = default
    lines.append(f"{name}={value}")
    os.environ[name] = value

# Anything else the notebook advertises (Qwen collection, GDRIVE_*, optional
# LLM keys) is copied through when attached, and simply skipped when not.
OPTIONAL = [
    "QWEN_ZILLIZ_URI", "QWEN_ZILLIZ_TOKEN", "QWEN_ZILLIZ_COLLECTION_NAME",
    "STORAGE_BACKEND", "GDRIVE_OAUTH_CREDENTIALS_JSON", "GDRIVE_FOLDER_ID",
    "GDRIVE_DRIVE_ID", "GDRIVE_SCOPES", "GDRIVE_URL_TEMPLATE",
    "UPLOAD_RAW_TO_STORAGE", "UPLOAD_CHUNKS_TO_STORAGE",
    "LOCAL_STORAGE_DIR", "IMAGEKIT_PUBLIC_KEY", "IMAGEKIT_PRIVATE_KEY",
    "IMAGEKIT_URL_ENDPOINT",
    "GEMINI_API_KEY", "LLM_API_KEY", "OPENAI_API_KEY", "LLM_BASE_URL", "LLM_MODEL",
    "INGEST_ENTITY", "INGEST_ENTITY_TYPE",
]
for name in OPTIONAL:
    try:
        value = secrets.get_secret(name)
    except Exception:
        continue
    if any(line.startswith(name + "=") for line in lines):
        continue
    lines.append(f"{name}={value}")
    os.environ[name] = value

os.makedirs("Footage-Engine", exist_ok=True)
with open("Footage-Engine/.env", "w") as fh:
    fh.write("\n".join(lines) + "\n")

# Values are secret; keys are not, so only the names are logged.
print("wrote .env with keys:", ", ".join(sorted(l.split("=")[0] for l in lines)))

In [ ]:
REPO = "https://github.com/ANNASBlackHat/Footage-Engine.git"

subprocess.run(["git", "clone", "--depth", "1", REPO, "Footage-Engine"], check=True)
os.chdir("Footage-Engine")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[video,gdrive,mcp]"], check=True)

import footage_engine

from footage_engine.config import get_settings

cfg = get_settings()
print("db        :", cfg.DATABASE_URL.split("@")[-1])
print("vector    :", cfg.VECTOR_STORE, "| backend:", cfg.EMBEDDING_BACKEND)
print("storage   :", cfg.STORAGE_BACKEND, "| raw:", cfg.UPLOAD_RAW_TO_STORAGE, "| chunks:", cfg.UPLOAD_CHUNKS_TO_STORAGE)
print("embed dim :", cfg.EMBEDDING_DIMENSION)

if cfg.STORAGE_BACKEND == "gdrive":
    from footage_engine.storage import get_storage_backend

    backend = get_storage_backend(cfg)
    who = backend.service.about().get(fields="user,storageQuota").execute()
    quota = who.get("storageQuota", {})
    print("drive as  :", who["user"]["emailAddress"])
    print("identity  :", backend.identity)
    print("quota     : limit=%s usage=%s" % (quota.get("limit"), quota.get("usage")))
    if backend.identity == "service_account":
        print("WARNING: a service account has storageQuota.limit == 0; Drive will reject uploads.")
else:
    print("WARNING: STORAGE_BACKEND is not gdrive; nothing will be uploaded to Drive.")

In [ ]:
# Pull the Netscape cookies.txt before any yt-dlp call. Kaggle has no browser
# profile, so YOUTUBE_COOKIES_FROM_BROWSER cannot work here.
import shutil

cookies_url = os.environ["YT_COOKIES_URL"]
COOKIE_PATH = "yt_cookie.txt"
subprocess.run(
    ["curl", "-sL", "--fail", "--max-time", "60", cookies_url, "-o", COOKIE_PATH],
    check=True,
)

# A captive portal or expired worker key returns 200 with an HTML body. Catch
# that here rather than letting yt-dlp fail with a baffling cookie error.
head = open(COOKIE_PATH, encoding="utf-8", errors="replace").read(400).lstrip()
if not head.startswith("# Netscape HTTP Cookie File") and ".youtube.com" not in head:
    print("---- first 200 bytes of the response ----")
    print(open(COOKIE_PATH, encoding="utf-8", errors="replace").read(200))
    raise SystemExit(
        "Cookie fetch did not return a cookies.txt. Check that YT_COOKIES_URL is "
        "current — if the worker's key was rotated, this is what a stale one looks like."
    )

cookie_bytes = os.path.getsize(COOKIE_PATH)
domains = {ln.split("\t")[0] for ln in open(COOKIE_PATH, encoding="utf-8", errors="replace") if ln.strip() and not ln.startswith("#")}
print(f"cookies.txt: {cookie_bytes} bytes, domains={sorted(d for d in domains if d)[:8]}")

# Config reads the path; the file itself is gitignored-equivalent (untracked,
# and this is an ephemeral Kaggle filesystem).
with open(".env", "a") as fh:
    fh.write(f"YOUTUBE_COOKIES={os.path.abspath(COOKIE_PATH)}\n")
os.environ["YOUTUBE_COOKIES"] = os.path.abspath(COOKIE_PATH)

In [ ]:
urls = [u.strip() for u in os.environ["INGEST_URLS"].split(",") if u.strip()]
entity = os.environ.get("INGEST_ENTITY") or None
entity_type = os.environ.get("INGEST_ENTITY_TYPE", "other")

print(f"ingesting {len(urls)} video(s)")
for i, url in enumerate(urls, 1):
    print(f"\n{'=' * 70}\n[{i}/{len(urls)}] {url}\n{'=' * 70}")
    argv = [sys.executable, "scripts/ingest_youtube_video.py", url]
    if entity:
        argv += ["--entity", entity, "--entity-type", entity_type]
    result = subprocess.run(argv)
    if result.returncode != 0:
        # Keep going: one bad URL (removed video, region block) should not throw
        # away the chunks already produced by the videos before it.
        print(f"!! ingest failed rc={result.returncode} for {url} — continuing")

print("\nall done")

In [ ]:
# Post-run sanity check: how much actually landed, so a silent failure is visible
# in the notebook rather than only in the DB.
from sqlalchemy import create_engine, text

engine = create_engine(cfg.DATABASE_URL)
with engine.connect() as conn:
    for table in ("media_items", "chunks", "jobs"):
        try:
            n = conn.execute(text(f"SELECT count(*) FROM {table}")).scalar()
            print(f"{table:14} {n}")
        except Exception as exc:
            print(f"{table:14} n/a ({type(exc).__name__})")

    try:
        rows = conn.execute(
            text("SELECT provider, status, storage_path FROM media_items ORDER BY ingested_at DESC LIMIT 5")
        ).fetchall()
        for r in rows:
            print(dict(r._mapping))
    except Exception:
        pass

print("\nNote: SQLite would show 0 here even after a successful run — its file does")
print("not survive the session. Use a shared Postgres DATABASE_URL to verify remotely.")